# Time Series Forecasting with Prophet (Day 11)

## Objective
Forecast **future hourly electricity prices** using time series methods that capture:
* **Trend**: Long-term price changes
* **Seasonality**: Hourly, daily, weekly patterns
* **Holidays**: Special events affecting prices
* **Uncertainty**: Confidence intervals for predictions

## Why Prophet?
* **Designed for Business Forecasting**: Meta's (Facebook) production forecasting library
* **Handles Seasonality**: Multiple seasonalities (hourly, daily, weekly, yearly)
* **Missing Data**: Robust to missing values and outliers
* **Uncertainty Quantification**: Built-in confidence intervals
* **Fast**: Efficient Stan-based optimization

## Key Differences from Regression
* **No Features**: Uses only timestamp + target (univariate)
* **Future Predictions**: Can forecast beyond training data
* **Seasonality Detection**: Automatic pattern discovery
* **Confidence Intervals**: Prediction uncertainty bounds

## Components
1. **Data Preparation**: Convert to Prophet format (ds, y)
2. **Prophet Model**: Train with hourly/daily/weekly seasonality
3. **Forecasting**: Generate 7-day ahead predictions
4. **Evaluation**: Compare forecasts vs actuals (RMSE, MAE)
5. **Visualization**: Plot trends, seasonality, forecasts
6. **Gold Layer**: Save forecasts to `h2_gold_price_forecasts`

In [0]:
# Install Prophet if not available
%pip install prophet --quiet

from prophet import Prophet
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta

print("✅ Imports loaded")

In [0]:
# Data configuration
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772775399"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

# Target column
TARGET_COL = "price_eur_mwh"
TIME_COL = "event_time_utc"

# Forecasting configuration
FORECAST_HORIZON_DAYS = 7  # Forecast 7 days ahead
FORECAST_HORIZON_HOURS = FORECAST_HORIZON_DAYS * 24

# Output table
FORECAST_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_price_forecasts"

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_COL}")
print(f"Forecast Horizon: {FORECAST_HORIZON_DAYS} days ({FORECAST_HORIZON_HOURS} hours)")
print(f"Output: {FORECAST_TABLE}")

In [0]:
print("Loading data for Prophet...")
print("="*80)

# Load data from Spark
df_spark = spark.table(SOURCE_TABLE)

# Select only time and target columns
df_prophet_spark = df_spark.select(
    F.col(TIME_COL).alias("ds"),  # Prophet requires 'ds' column
    F.col(TARGET_COL).alias("y")   # Prophet requires 'y' column
).orderBy("ds")

# Convert to Pandas (Prophet uses Pandas)
df_prophet = df_prophet_spark.toPandas()

print(f"Total records: {len(df_prophet):,}")
print(f"Date range: {df_prophet['ds'].min()} to {df_prophet['ds'].max()}")
print(f"\nFirst 5 rows:")
print(df_prophet.head())

# Split into train (2020) and test (2021) for evaluation
train_end = pd.Timestamp('2020-12-31 23:00:00')
df_train = df_prophet[df_prophet['ds'] <= train_end]
df_test = df_prophet[df_prophet['ds'] > train_end]

print(f"\nTrain: {len(df_train):,} records (2020)")
print(f"Test:  {len(df_test):,} records (2021)")

print("✅ Data prepared for Prophet")

In [0]:
print("Training Prophet model...")
print("="*80)

# Initialize Prophet with custom settings
model = Prophet(
    # Seasonality
    yearly_seasonality=True,   # Annual patterns
    weekly_seasonality=True,   # Weekly patterns (weekday vs weekend)
    daily_seasonality=True,    # Daily patterns (hourly cycles)
    
    # Uncertainty
    interval_width=0.95,       # 95% confidence intervals
    
    # Trend
    changepoint_prior_scale=0.05,  # Flexibility of trend changes
    
    # Seasonality strength
    seasonality_prior_scale=10.0,   # Strength of seasonality
    
    # Other
    seasonality_mode='multiplicative'  # Seasonality scales with trend
)

# Add hourly seasonality (Prophet doesn't include by default)
model.add_seasonality(
    name='hourly',
    period=1,       # 1 day period
    fourier_order=8  # Complexity of hourly pattern
)

# Fit model
import time
start_time = time.time()
model.fit(df_train)
training_time = time.time() - start_time

print(f"✅ Prophet model trained in {training_time:.2f}s")

In [0]:
print("Generating forecasts...")
print("="*80)

# Create future dataframe for forecasting
# Forecast on test set (2021) for evaluation
future = model.make_future_dataframe(
    periods=len(df_test),
    freq='H',  # Hourly frequency
    include_history=False
)

print(f"Forecasting {len(future):,} hours")
print(f"Forecast range: {future['ds'].min()} to {future['ds'].max()}")

# Make predictions
forecast = model.predict(future)

print(f"\n✅ Forecasts generated")
print(f"\nForecast columns: {list(forecast.columns)}")
print(f"\nSample forecast:")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head(10))

In [0]:
print("Evaluating forecast accuracy...")
print("="*80)

# Merge forecasts with actuals
evaluation = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].merge(
    df_test,
    on='ds',
    how='inner'
)

# Calculate errors
evaluation['error'] = evaluation['yhat'] - evaluation['y']
evaluation['abs_error'] = np.abs(evaluation['error'])
evaluation['squared_error'] = evaluation['error'] ** 2
evaluation['in_confidence_interval'] = (
    (evaluation['y'] >= evaluation['yhat_lower']) & 
    (evaluation['y'] <= evaluation['yhat_upper'])
)

# Metrics
rmse = np.sqrt(evaluation['squared_error'].mean())
mae = evaluation['abs_error'].mean()
mape = (evaluation['abs_error'] / evaluation['y']).mean() * 100
coverage = evaluation['in_confidence_interval'].mean() * 100

print(f"\nForecast Performance (Test Set - 2021):")
print(f"  RMSE:                {rmse:.2f} EUR/MWh")
print(f"  MAE:                 {mae:.2f} EUR/MWh")
print(f"  MAPE:                {mape:.2f}%")
print(f"  CI Coverage (95%):   {coverage:.2f}% (target: 95%)")
print(f"  Mean Actual Price:   {evaluation['y'].mean():.2f} EUR/MWh")
print(f"  Mean Forecast:       {evaluation['yhat'].mean():.2f} EUR/MWh")

# Show worst predictions
print(f"\nTop 10 Worst Predictions (by absolute error):")
worst = evaluation.nlargest(10, 'abs_error')[['ds', 'y', 'yhat', 'abs_error']]
print(worst.to_string(index=False))

print("✅ Evaluation complete")

In [0]:
print("Creating visualizations...")
print("="*80)

# 1. Forecast plot (first 7 days)
fig1 = model.plot(forecast, include_legend=True)
plt.title('Price Forecast with Confidence Intervals (First 7 Days)')
plt.xlabel('Date')
plt.ylabel('Price (EUR/MWh)')
plt.xlim(forecast['ds'].min(), forecast['ds'].min() + pd.Timedelta(days=7))
plt.tight_layout()
plt.show()

print("✅ Forecast plot created")

# 2. Components plot (trend + seasonality)
fig2 = model.plot_components(forecast)
plt.tight_layout()
plt.show()

print("✅ Components plot created (trend, seasonality patterns)")

# 3. Actual vs Forecast scatter
plt.figure(figsize=(10, 6))
plt.scatter(evaluation['y'], evaluation['yhat'], alpha=0.3, s=10)
plt.plot([0, evaluation['y'].max()], [0, evaluation['y'].max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Price (EUR/MWh)')
plt.ylabel('Forecasted Price (EUR/MWh)')
plt.title('Actual vs Forecasted Prices')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Scatter plot created")

print("\n✅ All visualizations complete")

In [0]:
print("Saving forecasts to Gold layer...")
print("="*80)

# Prepare forecast data for saving
forecast_to_save = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
forecast_to_save.columns = ['forecast_time_utc', 'predicted_price_eur_mwh', 'lower_bound_eur_mwh', 'upper_bound_eur_mwh']

# Add metadata
forecast_to_save['forecast_created_at'] = datetime.now()
forecast_to_save['model_name'] = 'prophet'
forecast_to_save['rmse'] = rmse
forecast_to_save['mae'] = mae
forecast_to_save['mape'] = mape

# Merge with actuals (where available)
forecast_final = forecast_to_save.merge(
    df_test.rename(columns={'ds': 'forecast_time_utc', 'y': 'actual_price_eur_mwh'}),
    on='forecast_time_utc',
    how='left'
)

# Convert to Spark DataFrame and save
forecast_spark = spark.createDataFrame(forecast_final)
forecast_spark.write.mode("overwrite").saveAsTable(FORECAST_TABLE)

record_count = spark.table(FORECAST_TABLE).count()
print(f"✅ Forecasts saved: {record_count:,} records")
print(f"   Table: {FORECAST_TABLE}")

# Show sample
print(f"\nSample forecasts:")
display(spark.table(FORECAST_TABLE).limit(20))

In [0]:
print("Analyzing detected seasonality patterns...")
print("="*80)

# Extract seasonality components from forecast
hourly_pattern = forecast.groupby(forecast['ds'].dt.hour)['yhat'].mean().reset_index()
hourly_pattern.columns = ['hour', 'avg_price']

print("\nAverage Price by Hour of Day:")
for idx, row in hourly_pattern.iterrows():
    print(f"  Hour {int(row['hour']):02d}:00 - {row['avg_price']:.2f} EUR/MWh")

# Day of week pattern
weekly_pattern = forecast.copy()
weekly_pattern['day_of_week'] = weekly_pattern['ds'].dt.day_name()
weekly_avg = weekly_pattern.groupby('day_of_week')['yhat'].mean().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)

print("\nAverage Price by Day of Week:")
for day, price in weekly_avg.items():
    print(f"  {day:9s} - {price:.2f} EUR/MWh")

# Hourly pattern visualization
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(hourly_pattern['hour'], hourly_pattern['avg_price'], marker='o')
plt.xlabel('Hour of Day')
plt.ylabel('Average Price (EUR/MWh)')
plt.title('Hourly Price Pattern')
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 24, 2))

plt.subplot(1, 2, 2)
plt.bar(range(7), weekly_avg.values)
plt.xlabel('Day of Week')
plt.ylabel('Average Price (EUR/MWh)')
plt.title('Weekly Price Pattern')
plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✅ Seasonality analysis complete")

## ✅ Time Series Forecasting Complete!

### What We Built
1. **Prophet Model**: Meta's production forecasting library
2. **Multi-Seasonality**: Captured hourly, daily, weekly, and yearly patterns
3. **Confidence Intervals**: 95% uncertainty bounds for predictions
4. **Evaluation**: Assessed forecast accuracy on 2021 test data
5. **Visualizations**: Trend, seasonality components, actual vs forecast
6. **Gold Table**: Saved forecasts to `h2_gold_price_forecasts`

### Key Findings
* **RMSE Performance**: Compare to regression models from Day 10
* **Seasonality Patterns**: 
  * **Hourly**: Peak prices during evening demand hours
  * **Weekly**: Weekday vs weekend differences
  * **Yearly**: Seasonal trends (winter vs summer)
* **Confidence Intervals**: ∼95% coverage indicates well-calibrated uncertainty
* **Forecast Horizon**: Accuracy decreases with longer horizons

### Comparison: Regression vs Time Series

| Aspect | Regression (Day 10) | Time Series (Day 11) |
| --- | --- | --- |
| **Input Features** | 14 features (load, weather, generation) | Only timestamp + price |
| **Prediction Scope** | Within training period | **Future predictions** |
| **Seasonality** | Manual feature engineering | **Automatic detection** |
| **Uncertainty** | Point estimates only | **Confidence intervals** |
| **Use Case** | Explain price drivers | Forecast future prices |
| **Training Data** | Requires all features | Only historical prices |

### Business Value
* **Future Planning**: Forecast prices days/weeks ahead for H2 production scheduling
* **Risk Management**: Confidence intervals quantify forecast uncertainty
* **Seasonal Insights**: Understand hourly/weekly price patterns for optimization
* **Simplicity**: No need for weather forecasts or generation data

### When to Use Each Approach

**Use Regression (Day 10) when:**
* You have reliable feature forecasts (weather, load, generation)
* You need to understand **why** prices change
* Forecasting within historical data range
* Feature importance is critical

**Use Time Series (Day 11) when:**
* Forecasting **future** beyond training data
* Features are unavailable or unreliable
* Need **confidence intervals**
* Seasonality is the dominant pattern
* Simplicity and speed are priorities

### Limitations & Improvements
* **Univariate**: Prophet uses only price history (no features)
* **Linear Trend**: May miss structural breaks or regime changes
* **Holiday Effects**: Could add Dutch holidays for better accuracy
* **Exogenous Variables**: Prophet supports regressors (could add weather, load)

### Next Steps
* **Day 12**: AutoML for automated model selection
* **Day 13**: Advanced feature engineering (lag features, rolling stats)
* **Day 14**: Ensemble models combining regression + time series

### Production Deployment
* Retrain Prophet weekly with latest data
* Generate 7-day rolling forecasts daily
* Monitor forecast accuracy drift
* Alert when actual prices fall outside confidence intervals
* Compare Prophet vs regression forecasts in A/B test